# m13 lab · Catheter design

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A catheter is a set of electrodes at chosen sizes and spacings, plus sensors. You compute how electrode size sets spreading resistance, how bipole direction sets amplitude, how electrode spacing and height above the tissue set spatial aliasing, how a basket's splines cover a chamber, and how three strain sensors estimate contact force.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Electrode size and spreading resistance | eq-m13-spreading |
| B | Bipole direction and orthogonal pairs | eq-m13-orthogonal |
| C | Spacing, height and spatial aliasing | eq-m13-alias |
| D | Basket coverage | eq-m13-basket |
| E | Contact force from three sensors | eq-m13-force, eq-m13-force-noise |

Lengths in mm, conductivity relative; forces and sensor readings relative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import catheters as K

## Part A · Electrode size and spreading resistance

A small electrode in blood sees a resistance to a distant return that is concentrated near the electrode: the
**spreading resistance**. For a disk flush with an insulating catheter body it is $1/(4\sigma a)$ (Newman); for a
hemisphere $1/(2\pi\sigma a)$; for a sphere in an infinite medium $1/(4\pi\sigma a)$ (m27). First check numerically that
the current density $1/\sqrt{a^2 - r^2}$ makes the disk an equipotential, then compare the shapes.

In [ ]:
radii = [0.0, 0.3, 0.6, 0.9, 0.99]
pots = [K.disk_density_potential(r, 1.0) for r in radii]
print("potential across the disk (should all be pi^2 = %.6f):" % np.pi**2, np.round(pots, 6))
a = np.linspace(0.25, 2.0, 50)
plt.plot(a, K.disk_resistance(a), label="disk (flush)")
plt.plot(a, K.hemisphere_resistance(a), label="hemisphere")
plt.plot(a, K.sphere_resistance(a), label="sphere, infinite medium")
plt.xlabel("electrode radius a (mm)"); plt.ylabel("resistance (1/(σ·mm))"); plt.legend(); plt.show()
print("disk / hemisphere at equal radius:", K.disk_resistance(1) / K.hemisphere_resistance(1))
print("numeric disk resistance, a = 2 mm:", K.disk_resistance_numeric(2.0), "closed form:", K.disk_resistance(2.0))

In [ ]:
# Checks
assert np.allclose(pots, np.pi**2, rtol=1e-8)
assert abs(K.disk_resistance_numeric(2.0) - K.disk_resistance(2.0)) < 1e-10
assert abs(K.disk_resistance(0.5) / K.disk_resistance(1.0) - 2) < 1e-12

## Part B · Bipole direction and orthogonal pairs

A closely spaced bipole records the potential gradient along its axis, so its amplitude falls as $|\cos\theta|$ with the
angle to the front's direction of travel (m03). A grid can form two perpendicular bipoles at one site. The better of the
two is never below $1/\sqrt2$ of the full amplitude; their root sum of squares is independent of direction.

In [ ]:
th = np.linspace(0, np.pi, 361)
best, rss = K.orthogonal_pair(th)
plt.plot(np.degrees(th), K.bipole_amplitude(th), label="one bipole")
plt.plot(np.degrees(th), best, label="better of two perpendicular")
plt.plot(np.degrees(th), rss, label="root sum of squares")
plt.xlabel("angle between bipole and propagation (degrees)"); plt.ylabel("amplitude (relative)"); plt.legend(); plt.show()
print("worst case, better of two: %.3f at %.0f degrees" % (best.min(), np.degrees(th[np.argmin(best)])))

In [ ]:
# Checks
assert abs(best.min() - 1/np.sqrt(2)) < 1e-4
assert np.allclose(rss, 1)

## Part C · Spacing, height and spatial aliasing

The sharpest signal a planar front makes across its width, seen from height $h$ above the activated layer, is the
Poisson kernel $(h/\pi)/(x^2+h^2)$, with spectrum $e^{-|k|h}$. Electrodes at spacing $\Delta$ capture wavenumbers up to
$\pi/\Delta$; the fraction of energy above that is $e^{-2\pi h/\Delta}$ (eq-m13-alias). Check it with an FFT, then build a
design table.

In [ ]:
for spacing, h in [(1.0, 1.0), (2.0, 1.0), (4.0, 1.0), (2.0, 0.5)]:
    print(f"spacing {spacing} mm, height {h} mm: formula {K.alias_fraction(spacing, h):.4f}, FFT {K.alias_fraction_fft(spacing, h):.4f}")
print()
print("largest spacing (mm) for a given energy above Nyquist")
print("height (mm)   10%    5%    1%")
for h in (0.5, 1.0, 2.0):
    print(f"   {h:4.1f}     " + "  ".join(f"{K.max_spacing(h, f):5.2f}" for f in (0.10, 0.05, 0.01)))

In [ ]:
x = np.linspace(-10, 10, 2001)
h = 1.0
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for spacing in (1.0, 3.0):
    xs = np.arange(-10 + spacing / 2, 10, spacing)
    ax[0].plot(xs, K.poisson_kernel(xs, h) / K.poisson_kernel(0, h), "o-", label=f"spacing {spacing} mm")
ax[0].plot(x, K.poisson_kernel(x, h) / K.poisson_kernel(0, h), "k", lw=1, label="true")
ax[0].set_xlabel("distance across the front (mm)"); ax[0].set_ylabel("signal (relative)"); ax[0].legend()
k = np.linspace(0, 8, 400)
ax[1].plot(k, np.exp(-k * h)); [ax[1].axvline(np.pi / s, ls="--", c=c) for s, c in ((1.0, "C0"), (3.0, "C1"))]
ax[1].set_xlabel("k (rad/mm)"); ax[1].set_ylabel("amplitude"); plt.show()

In [ ]:
# Checks
assert abs(K.alias_fraction_fft(2.0, 1.0) / K.alias_fraction(2.0, 1.0) - 1) < 0.01
assert abs(K.max_spacing(1.0, 0.01) - 1.364) < 1e-3
assert K.max_spacing(2.0, 0.01) == 2 * K.max_spacing(1.0, 0.01)

## Part D · Basket coverage

A basket's splines run pole to pole, so they are farthest apart at the equator: $2\pi R\sin\theta/n$ for $n$ evenly
spaced splines (eq-m13-basket). The **fill distance**, the largest distance from any point of the chamber wall to its
nearest electrode, measures coverage; here it is measured over the band the electrodes span (polar angles 25° to
155°), since no basket electrode reaches the poles. Compare an evenly spread basket with one whose splines bunch to one side, an
illustrative case of poor deployment. The 25 mm radius and the 8 × 8 layout are illustrative.

In [ ]:
R = 25.0
theta = np.radians(np.linspace(25, 155, 8))
print("interspline arc (mm) along each row:", np.round(K.interspline_arc(theta, 8, R), 1))
layouts = {"8 splines, even": K.basket(8, 8, R),
           "12 splines, even": K.basket(12, 8, R),
           "8 splines, bunched into 180 degrees": K.basket(8, 8, R, azimuths_deg=np.linspace(0, 180, 8))}
fill = {name: K.fill_distance(E, R, polar_range_deg=(25, 155)) for name, E in layouts.items()}
for name, d in fill.items():
    print(f"{name:38s} fill distance {d:5.1f} mm")

In [ ]:
# Checks
assert abs(K.interspline_arc(np.pi / 2, 8, R) - 2 * np.pi * R / 8) < 1e-12
assert fill["12 splines, even"] < fill["8 splines, even"] < fill["8 splines, bunched into 180 degrees"]

## Part E · Contact force from three sensors

Model three strain sensors around a spring element at angles $\varphi_i$: each reads $a F_z + b(F_x\cos\varphi_i +
F_y\sin\varphi_i)$ plus noise (eq-m13-force). Least squares gives the force; its covariance is $\sigma^2(C^\top C)^{-1}$
(eq-m13-force-noise). Compare that with Monte Carlo trials, then try uneven layouts.

In [ ]:
rng = np.random.default_rng(13)
C = K.force_matrix([0, 120, 240])
F = np.array([5.0, 0.0, 20.0])
est = K.force_trials(C, F, 1.0, 20000, rng)
print("mean estimate:", np.round(est.mean(0), 3))
print("Monte Carlo SD:", np.round(est.std(0), 3), " theory:", np.round(np.sqrt(np.diag(K.force_covariance(C, 1.0))), 3))
for beta in (60, 90, 120, 150):
    Cb = K.force_matrix([0, beta, -beta])
    sd = np.sqrt(np.diag(K.force_covariance(Cb, 1.0)))
    print(f"sensors at 0, ±{beta:3d} deg: SD Fx {sd[0]:.2f}, Fy {sd[1]:.2f}, Fz {sd[2]:.2f}; condition number {np.linalg.cond(Cb):.2f}")

In [ ]:
# Checks
sd_theory = np.sqrt(np.diag(K.force_covariance(C, 1.0)))
assert np.allclose(est.std(0), sd_theory, rtol=0.03)
assert np.allclose(sd_theory, [np.sqrt(2/3), np.sqrt(2/3), 1/np.sqrt(3)])

## Where to go next

- Add a fourth sensor and see how the covariance shrinks; then let one sensor fail.
- Replace the Poisson kernel by the m03 unipolar model of a front and repeat part C for unipolar signals.
- m14 turns spreading resistance into thermal noise; m23 shows how bipole direction and spacing shape voltage maps.